# Step 2: Feature Engineering

以 `reels_shortcode` 為 key，將各來源資料拼成兩張特徵表：
- `new_features.csv`：bass 224 筆 + static_info、embedding、ads_type
- `hist_features.csv`：hist_reels 10487 筆 + embedding、ads_type

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. 讀取原始資料

In [ ]:
hist_reels = pd.read_csv(RAW / 'hist_reels.csv')
bass       = pd.read_csv(RAW / 'bass_parameters_by_reel.csv')
static     = pd.read_csv(RAW / 'reels_static_info.csv')
ads        = pd.read_csv(RAW / 'ads_type_results.csv')

# embedding：只取指定欄位
emb_cols = ['reels_shortcode', 'Reels連結', 'KOL帳號', 'best_topic_labels'] + \
           [f'emb_{i:03d}' for i in range(32)]
embedding = pd.read_csv(RAW / 'reels_embedding.csv', usecols=emb_cols)

print('hist_reels :', hist_reels.shape)
print('bass       :', bass.shape)
print('static     :', static.shape)
print('ads        :', ads.shape)
print('embedding  :', embedding.shape)

## 2. 建立 new_features（bass 224 筆）

In [ ]:
new_features = (
    bass
    .merge(static,    on='reels_shortcode', how='left')
    .merge(embedding, on='reels_shortcode', how='left')
    .merge(ads,       on='reels_shortcode', how='left')
)

print('new_features shape :', new_features.shape)
print('\n欄位：')
print(new_features.columns.tolist())

## 3. 建立 hist_features（hist_reels 10487 筆）

In [ ]:
hist_features = (
    hist_reels
    .merge(embedding, on='reels_shortcode', how='left')
    .merge(ads,       on='reels_shortcode', how='left')
)

print('hist_features shape :', hist_features.shape)
print('\n欄位：')
print(hist_features.columns.tolist())

## 4. 檢查 merge 遺失情況

In [ ]:
print('=== new_features 各欄位缺值數 ===')
print(new_features.isnull().sum()[new_features.isnull().sum() > 0])

print('\n=== hist_features 各欄位缺值數 ===')
print(hist_features.isnull().sum()[hist_features.isnull().sum() > 0])

## 5. 存檔

In [ ]:
new_features.to_csv(PROCESSED / 'new_features.csv', index=False)
hist_features.to_csv(PROCESSED / 'hist_features.csv', index=False)

print('已儲存：')
print(f'  {PROCESSED / "new_features.csv"}')
print(f'  {PROCESSED / "hist_features.csv"}')